# KNN classification

In [1]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "numpy", "torch", "torchvision", "scikit-learn", "matplot"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



In [2]:
import random
from typing import Tuple, Dict
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

# Process data

In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def get_xray_loaders(
    dataDir: str,
    seed: int,
    batch_size: int = 32,
) -> Tuple[DataLoader, DataLoader]:
    set_seed(seed)
 
    transform = T.Compose([
        T.Resize((64, 64)),
        T.ToTensor()
    ])
 
    fullDataset = torchvision.datasets.ImageFolder(root=dataDir, transform=transform)
 
    trainIndices = []
    testIndices  = []
 
    for classIdx in range(len(fullDataset.classes)):
 
        classIndices = []
        for i in range(len(fullDataset.samples)):
            _, label = fullDataset.samples[i]
            if label == classIdx:
                classIndices.append(i)
 
        random.shuffle(classIndices)
 
        splitIdx = int(len(classIndices) * 0.7)
 
        trainIndices += classIndices[:splitIdx]
        testIndices  += classIndices[splitIdx:]
 
    trainDataset = Subset(fullDataset, trainIndices)
    testDataset  = Subset(fullDataset, testIndices)
 
    train_loader = DataLoader(trainDataset, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(testDataset,  batch_size=batch_size, shuffle=False)
 
    print(f"Train: {len(trainDataset)} | Test: {len(testDataset)}")
    return train_loader, test_loader

In [4]:
train_loader, test_loader = get_xray_loaders(dataDir="data/", seed=0)

Train: 3659 | Test: 1569


# KNN

In [5]:
def extract_features(dataloader):
    all_features, all_labels = [], []
    for images, labels in dataloader:
        # images shape: (batch, C, H, W)
        flat = images.view(images.size(0), -1).numpy()   # flatten each image
        all_features.append(flat)
        all_labels.append(labels.numpy())
    return np.vstack(all_features), np.concatenate(all_labels)

print("Extracting features...")
X_train, y_train = extract_features(train_loader)
X_test,  y_test  = extract_features(test_loader)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")


Extracting features...
Train: (3659, 12288)  |  Test: (1569, 12288)


In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import time

K = [1, 3, 5, 10]
for k in K:
    knn = KNeighborsClassifier(n_neighbors=k, metric="euclidean", n_jobs=-1)
    knn.fit(X_train, y_train)
    start = time.perf_counter()
    y_pred = knn.predict(X_test)
    elapsed = time.perf_counter() - start

    print(f"\nAccuracy for {k} neigbors: {accuracy_score(y_test, y_pred):.4f}")
    print(f"\nRuntime: {elapsed} seconds")


Accuracy for 1 neigbors: 0.9273

Runtime: 4.378621700016083 seconds

Accuracy for 3 neigbors: 0.9293

Runtime: 1.7124688999901991 seconds

Accuracy for 5 neigbors: 0.9293

Runtime: 1.6079938000184484 seconds

Accuracy for 10 neigbors: 0.9305

Runtime: 1.4756688000052236 seconds


# With PCA

In [9]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca_full = PCA().fit(X_train)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_95   = np.searchsorted(cumvar, 0.95) + 1
n_99   = np.searchsorted(cumvar, 0.99) + 1

N_COMPONENTS = n_95
pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

K = [1, 3, 5, 10]
for k in K:
    knn_pca = KNeighborsClassifier(n_neighbors=k, metric="euclidean", n_jobs=-1)
    knn_pca.fit(X_train_pca, y_train)

    y_pred_pca = knn_pca.predict(X_test_pca)

    acc_raw = accuracy_score(y_test, y_pred)
    print(f"\nData for {k} neighbors")
    print(f"Accuracy (raw): {acc_raw}")

    acc_pca = accuracy_score(y_test, y_pred_pca)
    print(f"Accuracy (PCA): {acc_pca}")
    print(f"Difference: {acc_pca - acc_raw}")

    start = time.perf_counter()
    knn.predict(X_test)
    t_raw = (time.perf_counter() - start)

    start = time.perf_counter()
    knn_pca.predict(X_test_pca)
    t_pca = (time.perf_counter() - start)

    print(f"\nRuntime (raw) : {t_raw} seconds")
    print(f"Runtime (PCA) : {t_pca} seconds")
    print(f"Speedup: {t_raw / t_pca}x")


Data for 1 neighbors
Accuracy (raw): 0.9305289993626513
Accuracy (PCA): 0.9279796048438496
Difference: -0.002549394518801762

Runtime (raw) : 1.2200205000117421 seconds
Runtime (PCA) : 0.021640100021613762 seconds
Speedup: 56.37776622072941x

Data for 3 neighbors
Accuracy (raw): 0.9305289993626513
Accuracy (PCA): 0.9375398342893563
Difference: 0.007010834926704956

Runtime (raw) : 1.2149270999943838 seconds
Runtime (PCA) : 0.02546570001868531 seconds
Speedup: 47.708372402994534x

Data for 5 neighbors
Accuracy (raw): 0.9305289993626513
Accuracy (PCA): 0.935627788400255
Difference: 0.005098789037603635

Runtime (raw) : 1.2347527999954764 seconds
Runtime (PCA) : 0.02392959999269806 seconds
Speedup: 51.599391564098525x

Data for 10 neighbors
Accuracy (raw): 0.9305289993626513
Accuracy (PCA): 0.9330783938814532
Difference: 0.002549394518801873

Runtime (raw) : 1.190262200019788 seconds
Runtime (PCA) : 0.026089899998623878 seconds
Speedup: 45.62157003601274x
